# 05 — Valutazione e analisi dei casi di errore
Metriche di rilevamento (mAP, precision, recall) + casi di errore qualitativi.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from src.config import set_reproducibility, select_device, SH17_CLASSES
set_reproducibility(42); print('device:', select_device('auto'))

In [ ]:
from src.evaluate import evaluate_yolo
summary = evaluate_yolo(weights=str(root / 'models/best.pt'), data=str(root / 'data/processed/sh17/data.yaml'),
    device='auto', split='val')
summary

## Casi di errore
Esegui l'inferenza su una manciata di immagini, salva quelle su cui il modello sbaglia in
`outputs/failure_cases/` e categorizzale in `reports/failure_analysis.md`.

In [ ]:
import cv2
from src.inference import load_model, predict_image
from src.compliance import assess_compliance
from src.visualization import draw_detections, draw_compliance
from src.data import list_images

model = load_model(root / 'models/best.pt')
(root / 'outputs/failure_cases').mkdir(parents=True, exist_ok=True)
for p in list_images(root / 'data/processed/sh17/images/test')[:20]:
    im = cv2.imread(str(p)); dets = predict_image(model, im, conf=0.25)
    out = draw_compliance(draw_detections(im, dets, SH17_CLASSES), assess_compliance(dets))
    cv2.imwrite(str(root / f'outputs/failure_cases/{p.stem}.jpg'), out)
print('saved candidate cases to outputs/failure_cases/')